In [2]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\admin\AppData\Local\Temp\ipykernel_1800\2148886914.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [4]:
DATA_DIR = Path("data")

In [5]:
persist_directory = DATA_DIR / "bookstore"

In [6]:
pdf_files_paths = DATA_DIR.glob("*.pdf")

In [7]:
list(pdf_files_paths)

[WindowsPath('data/Atomic habits ( PDFDrive ).pdf'),
 WindowsPath('data/attention.pdf'),
 WindowsPath('data/BhagavadGita.pdf'),
 WindowsPath('data/the-5-am-club.pdf')]

In [8]:
def normalize_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text.lower()).strip("_")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()

In [9]:
all_documents = []
source_ids = set() 

for pdf_file_path in DATA_DIR.glob("*.pdf"):
    print(f"{pdf_file_path.name}: {file_sha256(pdf_file_path)}")
    loader = PyMuPDFLoader(str(pdf_file_path))
    documents = loader.load()

    source_checksum = file_sha256(pdf_file_path)
    source_id = normalize_name(pdf_file_path.stem)
    source_ids.add(source_id)

    for index, doc in enumerate(documents):
        doc.metadata["source"] = pdf_file_path.name
        doc.metadata["source_id"] = source_id
        doc.metadata["source_checksum"] = source_checksum
        doc.metadata["page_number"] = index + 1
        doc.metadata["chunk_id"] = str(uuid4())
        doc.metadata["chunk_index"] = index
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["created_at"] = datetime.now(timezone.utc).isoformat()

    all_documents.extend(documents)

documents = all_documents
print(len(documents), "documents loaded from all PDFs.")
sorted(set(source_ids))

Atomic habits ( PDFDrive ).pdf: a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878
attention.pdf: bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697
BhagavadGita.pdf: ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583
the-5-am-club.pdf: 089bb3fc5b41e7de713444b93214434e1d887c47dff66e8e4ae075aeed89440b
1475 documents loaded from all PDFs.


['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

In [11]:
chunks = splitter.split_documents(documents)
chunks[0].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits \\( PDFDrive.com \\).pdf',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 2,
 'source_id': 'atomic_habits___pdfdrive',
 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'page_number': 3,
 'chunk_id': 'b6c32814-8506-4b88-98a7-a73fd168a4a2',
 'chunk_index': 2,
 'chunk_size': 531,
 'created_at': '2026-07-29T14:00:31.933127+00:00'}

In [12]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [13]:
collection_name = "bookstore"
collection_name

'bookstore'

In [ ]:
if not persist_directory.exists():
    persist_directory.mkdir(parents=True, exist_ok=True)

temp_store = Chroma(
    persist_directory=str(persist_directory),
    embedding_function=embeddings,
    collection_name=collection_name,
    )

try:
    temp_store._client.delete_collection(name=collection_name)
except Exception:
    pass

vector_store = Chroma(
    persist_directory=str(persist_directory),
    embedding_function=embeddings,
    collection_name=collection_name,
    )

vector_store

In [15]:
inserted_count = 0
batch_size = 100

for start in range(0, len(chunks), batch_size):
    end = start + batch_size
    vector_store.add_documents(chunks[start:end])
    inserted_count += len(chunks[start:end])

print("Inserted chunks:", inserted_count)

Inserted chunks: 3583


# Creating retriever for the RAG system

In [16]:
query = ['What does krishna says about the Mahabharat battle']

In [17]:
vector_store

In [18]:
vector_store

In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 10, "filter": {"source_id": "bhagavadgita"}}
)

In [20]:
retriever.invoke(input=query[0])

[Document(id='1417295d-6db1-4bdc-ba5f-25d68dea9390', metadata={'page': 50, 'creator': 'Pages', 'page_number': 51, 'file_path': 'data\\BhagavadGita.pdf', 'moddate': "D:20120508142201Z00'00'", 'format': 'PDF 1.3', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'author': 'me', 'creationdate': "D:20120508142201Z00'00'", 'modDate': "D:20120508142201Z00'00'", 'subject': '', 'source': 'BhagavadGita.pdf', 'title': 'Bhagavad-gita As It Is with pics!', 'chunk_index': 50, 'total_pages': 952, 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'chunk_id': '2e941187-4174-4f60-8f36-b120b765d88e', 'created_at': '2026-07-29T14:00:36.843091+00:00', 'trapped': '', 'source_id': 'bhagavadgita', 'chunk_size': 1333, 'creationDate': "D:20120508142201Z00'00'", 'keywords': ''}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

# Building a Part 2 - RAG Pipeline

## Creating Stuff Documents Chain

In [21]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [22]:
from langchain_core.prompts import ChatPromptTemplate

In [23]:
system_prompt = '''

You are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.
Your task is to provide a concise and accurate answer to the question using the information from the documents. 
If the answer is not present in the documents, respond with "I don't know."

context:{context}

'''

In [24]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [25]:
from langchain.chat_models.base import init_chat_model

In [26]:
llm = init_chat_model(
    model='gpt-5.4-nano',
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0.0,
    max_tokens=512,
) 

In [27]:
response = llm.invoke("Hi")
response.content


'Hi! 👋 How can I help you today?'

In [28]:
combine_docs_chain = create_stuff_documents_chain(
    llm, prompt
)

## Creating Retrival Chain

In [29]:
from langchain_classic.chains import create_retrieval_chain

In [30]:
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [31]:
result = rag_chain.invoke({'input': 'What did Krishna say about the Mahabharat battle?'})
result['answer']

'From the provided text, Kåñëa said that the battle is unavoidable because **Time Himself is destroying everything**, and He has come **to engage all people**. He also stated that **“with the exception of the Pāṇḍavas,” all the soldiers on both sides will be slain** (Text 33: “Time I am, destroyer of the worlds…”).'

# Adding Hybrid Search

In [32]:
retriever  # Dense retriever for semantic search in the shared vector store.

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000013717C275F0>, search_kwargs={'k': 10, 'filter': {'source_id': 'bhagavadgita'}})

### Adding Dense + Sparse to Build Hybrid Search

In [33]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document


In [34]:
chunks[0]

Document(metadata={'producer': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creationdate': '2020-04-30T18:46:22+00:00', 'source': 'Atomic habits ( PDFDrive ).pdf', 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf', 'total_pages': 256, 'format': 'PDF 1.4', 'title': 'Atomic habits \\( PDFDrive.com \\).pdf', 'author': 'James Clear', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20200430184622+00'00'", 'page': 2, 'source_id': 'atomic_habits___pdfdrive', 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878', 'page_number': 3, 'chunk_id': 'b6c32814-8506-4b88-98a7-a73fd168a4a2', 'chunk_index': 2, 'chunk_size': 531, 'created_at': '2026-07-29T14:00:31.933127+00:00'}, page_content='AN IMPRINT OF PENGUIN RANDOM HOUSE LLC\n375 Hudson Street\nNew York, New York 10014\nCopyright © 2018 by James Clear\nPenguin supports copyright. Copyright fuels creativity, enc

In [35]:
sparse_retriever = BM25Retriever.from_documents(chunks, k=5)
sparse_retriever

BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000013718394320>, k=5)

### Why combine dense and sparse retrieval?

- Dense retrieval uses embeddings to find chunks that are semantically similar to the query.
- Sparse retrieval with BM25 looks for exact or near-exact keyword overlap.
- A hybrid retriever combines both so the search is stronger on paraphrased questions and on questions with important names, terms, or dates.
- In this notebook, the two retrievers are given equal weight first so neither side dominates.
- What to observe: the hybrid retriever should return a more balanced set of chunks than either retriever alone.

In [36]:
from langchain_classic.retrievers import EnsembleRetriever

In [37]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[retriever, sparse_retriever],
    weights=[0.5, 0.5]
 )
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000013717C275F0>, search_kwargs={'k': 10, 'filter': {'source_id': 'bhagavadgita'}}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x0000013718394320>, k=5)], weights=[0.5, 0.5])

### How to read the hybrid retriever output

- `weights=[0.5, 0.5]` means both retrievers contribute equally to the final ranking.
- Raise the dense weight when the query is more conceptual or paraphrased.
- Raise the sparse weight when exact words, names, or technical terms matter most.
- The output documents are merged and ranked together before they are passed into the RAG chain.
- What to observe: compare the `doc_res` from `retriever` with the `doc_res` from `hybrid_retriever` to see how the candidate chunks change.

In [38]:
doc_res = retriever.invoke('input=What did Krishna say about the Mahabharat battle?')

In [39]:
for doc in doc_res:
    print(f"Source: {doc.metadata['source']}, Chunk index: {doc.metadata['chunk_index']}")

Source: BhagavadGita.pdf, Chunk index: 50
Source: BhagavadGita.pdf, Chunk index: 301
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 128
Source: BhagavadGita.pdf, Chunk index: 39
Source: BhagavadGita.pdf, Chunk index: 89
Source: BhagavadGita.pdf, Chunk index: 896
Source: BhagavadGita.pdf, Chunk index: 602
Source: BhagavadGita.pdf, Chunk index: 94


In [40]:
doc_res = hybrid_retriever.invoke('input=What did Krishna say about the Mahabharat battle?')

In [41]:
for doc in doc_res:
    print(f"Source: {doc.metadata['source']}, Chunk index: {doc.metadata['chunk_index']}")

Source: BhagavadGita.pdf, Chunk index: 50
Source: BhagavadGita.pdf, Chunk index: 911
Source: BhagavadGita.pdf, Chunk index: 301
Source: BhagavadGita.pdf, Chunk index: 910
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 940
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 903
Source: BhagavadGita.pdf, Chunk index: 128
Source: BhagavadGita.pdf, Chunk index: 951
Source: BhagavadGita.pdf, Chunk index: 39
Source: BhagavadGita.pdf, Chunk index: 89
Source: BhagavadGita.pdf, Chunk index: 896
Source: BhagavadGita.pdf, Chunk index: 602
Source: BhagavadGita.pdf, Chunk index: 94


In [42]:
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000013717CBCF20>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000137183F4F80>, root_client=<openai.OpenAI objec

In [43]:
combine_docs_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n\nYou are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.\nYour task is to provide a concise and accurate answer to the question using the information from the documents. \nIf the answer is not present in the documents, respond with "I don\'t know."\n\ncontext:{context}\n\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_

In [44]:
hybrid_retriever_chain = create_retrieval_chain(hybrid_retriever, combine_docs_chain)

In [45]:
response = hybrid_retriever_chain.invoke({"input": "What did Krishna say about the Mahabharat battle?"})

In [46]:
response['answer']

'The documents explain that Krishna told Arjuna that the fight was already determined by Him as the “destroyer of the worlds.” Specifically, Krishna said (Bhagavad-gītā 11.33):\n\n- **“Time I am, destroyer of the worlds, and I have come to engage all people.”**\n- **“With the exception of you [the Pāṇḍavas], all the soldiers here on both sides will be slain.”**'

# Adding Reranking

Retriever -> Rerank chain (prompt + LLM + parser) -> Top-k reranked documents -> Retrieval chain with CSDC -> Final answer

In [49]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000013717C275F0>, search_kwargs={'k': 10, 'filter': {'source_id': 'bhagavadgita'}})

Use this production rule:

Use PromptTemplate when:
You want one flat text prompt.
Output is simple text you will parse yourself (like rerank indices, JSON string, labels).
Typical chain: prompt | llm | parser
Use ChatPromptTemplate when:
You want role-based control (system/human/assistant).
You are building conversational QA/agent behavior.
You need stronger instruction hierarchy and safer behavior in multi-turn flows.

In [93]:
from langchain_core.prompts import PromptTemplate

rerank_prompt = PromptTemplate.from_template("""
You are a reranking assistant.

Task: Rank the MOST relevant documents to the user question.

User Question: {question}

Documents (numbered 1..N):
{documents}

Return ONLY valid JSON in this exact schema:
{{
  "ranked": [
    {{"rank": 1, "doc_index": 2, "reason": "short reason"}}
  ]
}}

Rules:
- Return exactly top 5 items.
- rank must be unique integers 1..5.
- doc_index must reference the numbered documents above (1-based index).
- Include each document at most once.
- reason must be <= 20 words.
- Do not include any text outside the JSON object.
""")

In [86]:
retrieved_docs=retriever.invoke('What does krishna says about the Mahabharat battle?')
retrieved_docs

[Document(id='1417295d-6db1-4bdc-ba5f-25d68dea9390', metadata={'chunk_index': 50, 'creationDate': "D:20120508142201Z00'00'", 'source_id': 'bhagavadgita', 'format': 'PDF 1.3', 'chunk_id': '2e941187-4174-4f60-8f36-b120b765d88e', 'creationdate': "D:20120508142201Z00'00'", 'author': 'me', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'chunk_size': 1333, 'page_number': 51, 'title': 'Bhagavad-gita As It Is with pics!', 'keywords': '', 'moddate': "D:20120508142201Z00'00'", 'creator': 'Pages', 'total_pages': 952, 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'trapped': '', 'file_path': 'data\\BhagavadGita.pdf', 'created_at': '2026-07-29T14:00:36.843091+00:00', 'source': 'BhagavadGita.pdf', 'page': 50, 'subject': '', 'modDate': "D:20120508142201Z00'00'"}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

In [87]:
for doc in retrieved_docs:
    print(f"Source: {doc.metadata['source']}, Chunk index: {doc.metadata['chunk_index']}")

Source: BhagavadGita.pdf, Chunk index: 50
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 901
Source: BhagavadGita.pdf, Chunk index: 301
Source: BhagavadGita.pdf, Chunk index: 884
Source: BhagavadGita.pdf, Chunk index: 135
Source: BhagavadGita.pdf, Chunk index: 128
Source: BhagavadGita.pdf, Chunk index: 602
Source: BhagavadGita.pdf, Chunk index: 896
Source: BhagavadGita.pdf, Chunk index: 603


When to Use Each in RAG
Use prompt | llm | StrOutputParser() when:
You need a custom text-to-text step.
Output is raw text you will parse yourself.
You want full control over prompt shape and parsing logic.
Typical use cases:
Reranking index output like 2,1,3,0
Query rewriting
Classification labels
Simple extraction
Use create_stuff_documents_chain(...) when:
You already have Document objects.
Goal is final grounded answer generation from retrieved context.
You want LangChain to combine documents into context and run QA.
Typical use cases:
Final answer stage in RAG
Context-grounded summarization
Citation-ready answer generation (with your own formatting layer)
Production RAG Pattern
Retrieve top-k candidates
Rerank with prompt | llm | StrOutputParser()
Select top-n reranked docs
Generate final answer with create_stuff_documents_chain(...)
One-Line Rule
Custom transformation step: prompt | llm | StrOutputParser()
Final context-grounded answering step: create_stuff_documents_chain(...)

In [88]:
from langchain_core.output_parsers import StrOutputParser 

In [94]:
rerank_chain=rerank_prompt| llm | StrOutputParser()
rerank_chain

PromptTemplate(input_variables=['documents', 'question'], input_types={}, partial_variables={}, template='\nYou are a reranking assistant.\n\nTask: Rank the MOST relevant documents to the user question.\n\nUser Question: {question}\n\nDocuments (numbered 1..N):\n{documents}\n\nReturn ONLY valid JSON in this exact schema:\n{{\n  "ranked": [\n    {{"rank": 1, "doc_index": 2, "reason": "short reason"}}\n  ]\n}}\n\nRules:\n- Return exactly top 5 items.\n- rank must be unique integers 1..5.\n- doc_index must reference the numbered documents above (1-based index).\n- Include each document at most once.\n- reason must be <= 20 words.\n- Do not include any text outside the JSON object.\n')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.14', 'langchain-openai': '1.3.5'}}, output_version=None, profile={'name': 'GPT-5.4 nano', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128

In [97]:
retrieved_docs #this were thre returned docs - clean and format for input to the rerank chain

[Document(id='1417295d-6db1-4bdc-ba5f-25d68dea9390', metadata={'chunk_index': 50, 'creationDate': "D:20120508142201Z00'00'", 'source_id': 'bhagavadgita', 'format': 'PDF 1.3', 'chunk_id': '2e941187-4174-4f60-8f36-b120b765d88e', 'creationdate': "D:20120508142201Z00'00'", 'author': 'me', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583', 'chunk_size': 1333, 'page_number': 51, 'title': 'Bhagavad-gita As It Is with pics!', 'keywords': '', 'moddate': "D:20120508142201Z00'00'", 'creator': 'Pages', 'total_pages': 952, 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'trapped': '', 'file_path': 'data\\BhagavadGita.pdf', 'created_at': '2026-07-29T14:00:36.843091+00:00', 'source': 'BhagavadGita.pdf', 'page': 50, 'subject': '', 'modDate': "D:20120508142201Z00'00'"}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

In [90]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
formatted_docs = "\n".join(doc_lines)

In [98]:
doc_lines

['1. position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme Lord Kåñëa was on the other side. But still, it was \nhis duty to conduct the fight, and no pains would be spared in that \nconnection.\nTEXT 13\nTaTa" Xa«aê >aeYaRê Pa<avaNak-GaaeMau%a" )\nSahSaEva>YahNYaNTa Sa XaBdSTauMaul/ae_>avTa( )) 13 ))',
 '2. would be transferred to Yudhiñöhira. It is also predicted here that Yudhiñöhira, \nafter gaining victory in this battle, would flourish more and more because he \nwas not only righteous and pious, but he was a strict moralist. He never spoke a \nlie during his life.\nThere are many less intelligent persons who take Bhagavad-gétä to be a \ndiscussion of topics between two friends in a battlefield. But such a book \ncannot be scripture. Some may protest that Kåñëa incited Arjuna to fight, \nwhich is immoral, but the reality of the situation is clearly st

In [99]:
formatted_docs

'1. position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme Lord Kåñëa was on the other side. But still, it was \nhis duty to conduct the fight, and no pains would be spared in that \nconnection.\nTEXT 13\nTaTa" Xa«aê >aeYaRê Pa<avaNak-GaaeMau%a" )\nSahSaEva>YahNYaNTa Sa XaBdSTauMaul/ae_>avTa( )) 13 ))\n2. would be transferred to Yudhiñöhira. It is also predicted here that Yudhiñöhira, \nafter gaining victory in this battle, would flourish more and more because he \nwas not only righteous and pious, but he was a strict moralist. He never spoke a \nlie during his life.\nThere are many less intelligent persons who take Bhagavad-gétä to be a \ndiscussion of topics between two friends in a battlefield. But such a book \ncannot be scripture. Some may protest that Kåñëa incited Arjuna to fight, \nwhich is immoral, but the reality of the situation is clearly stated

In [95]:
rerank_response = rerank_chain.invoke({
    "question": "What does krishna says about the Mahabharat battle?",
    "documents": formatted_docs
})
rerank_response

'{\n  "ranked": [\n    {"rank": 1, "doc_index": 6, "reason": "Krishna explains Arjuna’s duty; killing bodies doesn’t kill the soul."},\n    {"rank": 2, "doc_index": 5, "reason": "Krishna tells Arjuna to surrender and follow the Supersoul’s order to fight."},\n    {"rank": 3, "doc_index": 7, "reason": "Krishna says happy are fighters; war opportunities come by divine arrangement."},\n    {"rank": 4, "doc_index": 10, "reason": "Krishna’s plan governs Kurukshetra battle; Arjuna must desire the Supreme while fighting."},\n    {"rank": 5, "doc_index": 3, "reason": "Battle outcome: Kåñëa and Arjuna ensure victory for Yudhishthira’s side."}\n  ]\n}'

In [96]:
import json

# Parse reranker JSON output and attach original document content/metadata.
payload = json.loads(str(rerank_response))
ranked = payload.get("ranked", [])

reranked_with_docs = []
for item in ranked:
    idx_1_based = int(item.get("doc_index", 0))
    idx = idx_1_based - 1
    if 0 <= idx < len(retrieved_docs):
        doc = retrieved_docs[idx]
        reranked_with_docs.append({
            "rank": int(item.get("rank", len(reranked_with_docs) + 1)),
            "doc_index": idx_1_based,
            "reason": item.get("reason", ""),
            "source": doc.metadata.get("source"),
            "chunk_index": doc.metadata.get("chunk_index"),
            "page_number": doc.metadata.get("page_number"),
            "document": doc.page_content,
        })

reranked_with_docs

[{'rank': 1,
  'doc_index': 6,
  'reason': 'Krishna explains Arjuna’s duty; killing bodies doesn’t kill the soul.',
  'source': 'BhagavadGita.pdf',
  'chunk_index': 135,
  'page_number': 136,
  'document': 'Forgetting his prime duty, he wanted to cease fighting because he thought that \nby not killing his relatives and kinsmen he would be happier than by enjoying \nthe kingdom by conquering his cousins and brothers, the sons of Dhåtaräñöra. \nIn both ways, the basic principles were for sense gratification. Happiness \nderived from conquering them and happiness derived by seeing kinsmen alive \nare both on the basis of persona1 sense gratification, for there is a sacrifice of \nwisdom and duty. Kåñëa, therefore, wanted to explain to Arjuna that by killing \nthe body of his grandfather he would not be killing the soul proper, and He \nexplained that all individual persons, including the Lord Himself, are eternal \nindividuals; they were individuals in the past, they are individuals in th

## Rerank + CSDC + CRC (production flow)

This adds a rerank-aware retriever runnable and then uses:
- CSDC: `create_stuff_documents_chain(...)`
- CRC: `create_retrieval_chain(...)`

In [101]:
from langchain_core.runnables import RunnableLambda
from langchain_core.documents import Document


def rerank_retrieve(inputs: dict):
    question = inputs["input"]

    # 1) Retrieve candidates first (hybrid for better recall)
    candidate_docs = hybrid_retriever.invoke(question)

    # 2) Prepare numbered docs for reranker
    candidate_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(candidate_docs)]
    candidate_text = "\n".join(candidate_lines)

    # 3) Rerank with strict JSON output; fallback safely if parsing fails
    rerank_raw = rerank_chain.invoke({"question": question, "documents": candidate_text})

    try:
        rerank_payload = json.loads(str(rerank_raw))
        ranked_items = rerank_payload.get("ranked", [])
    except Exception:
        ranked_items = []

    reranked_docs = []
    for item in ranked_items:
        idx_1_based = int(item.get("doc_index", 0))
        idx = idx_1_based - 1
        if 0 <= idx < len(candidate_docs):
            reranked_docs.append(candidate_docs[idx])

    # Keep top-5 and ensure we always return docs for answer generation
    if not reranked_docs:
        reranked_docs = candidate_docs[:5]
    else:
        reranked_docs = reranked_docs[:5]

    return reranked_docs


rerank_retriever = RunnableLambda(rerank_retrieve)
rerank_retriever

RunnableLambda(rerank_retrieve)

In [102]:
# CSDC: combine retrieved docs into answer context
rerank_csdc = create_stuff_documents_chain(llm, prompt)

# CRC: retrieval + CSDC answer generation
rerank_crc = create_retrieval_chain(rerank_retriever, rerank_csdc)

rerank_crc

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(rerank_retrieve), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n\nYou are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.\nYour task is to provide a concise and accurate answer to the question using the information from the documents. \nIf the answer is not present in the documents, res

In [103]:
final_rerank_result = rerank_crc.invoke({"input": "What did Krishna say about the Mahabharat battle?"})
final_rerank_result["answer"]

'Krishna told Arjuna that the battle was already decided because He was on the side of the Pāṇḍavas. He said that He is “Time… destroyer of the worlds” and that He had come “to engage all people,” adding that “with the exception of you (the Pāṇḍavas), all the soldiers here on both sides will be slain.”'